# Breaking a Spam Filter

A simple spam classifier (TF-IDF + Naive Bayes) trained on the
[SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) dataset.
Goal: train it, measure it, then write messages that fool it and explain why.

## 1. Load the data

The file has one message per line: the label (`ham` or `spam`), a tab, then the message.
It has no header row, so we name the columns ourselves.
The dataset has 5,574 messages, of which 747 (13.4%) are spam.

**Baseline:** a "model" that always answers *not spam* would already be 86.6% accurate,
so our model has to beat that.

In [1]:
import pandas as pd
import csv

df = pd.read_csv("data/SMSSpamCollection", 
                 sep="\t", 
                 header=None,
                 names=["label", "text"],
                 quoting=csv.QUOTE_NONE)

print(df.shape)
df.head()

(5574, 2)


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 2. Split into training and test sets

- `x` = the messages (what the model reads), `y` = the labels (the answers)
- 80% is used for training; 20% is hidden as the test set, so we can check the model on messages it has never seen
- `stratify=y` keeps the same share of spam (13.4%) in both sets

In [8]:
from sklearn.model_selection import train_test_split

x = df["text"]
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

## 3. Turn text into numbers (TF-IDF)

A model can only work with numbers. `TfidfVectorizer` builds a vocabulary from the training messages
and turns each message into a row with one column per word (7,716 words), giving rare words a higher score.

- `fit_transform` on the training data: learn the vocabulary, then convert
- `transform` on the test data: convert using the same vocabulary (no peeking at the test set)

Words that are not in the training vocabulary are ignored.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()

x_train_vec = vec.fit_transform(x_train)
x_test_vec = vec.transform(x_test)

print(x_train_vec.shape, x_test_vec.shape)
print(len(vec.get_feature_names_out()))

(4459, 7716) (1115, 7716)
7716


## 4. Train the model

`MultinomialNB` (Naive Bayes) learns, for each word, how strongly it points towards spam or not spam.
New messages must go through `vec.transform` first, because the model only understands the 7,716 TF-IDF columns.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(x_train_vec, y_train)


['spam' 'ham']


## 5. Measure the model

| | Accuracy |
|---|---|
| Baseline (always "not spam") | 86.6% |
| Train | 97.4% |
| Test | 96.2% |

**Better than the baseline?** ___ (by how much, and what that means)

**Overfitting, underfitting or good fit?** ___ (compare train and test, and explain)

In [ ]:
from sklearn.metrics import accuracy_score
pred_train = model.predict(x_train_vec)
pred_test = model.predict(x_test_vec)

print("train accuracy: ", accuracy_score(y_train, pred_train))
print("test accuracy: ", accuracy_score(y_test, pred_test))


train accuracy:  0.9744337295357703
train accuracy:  0.9623318385650225
